To get the full picture of Python's type system — both the runtime object model and the static-typing layer bolted on top — you need a handful of standard-library modules. Here's the set, roughly in order of how foundational they are.

**`builtins`.** The implicit module everything starts from: `int`, `float`, `str`, `bytes`, `bytearray`, `list`, `tuple`, `dict`, `set`, `frozenset`, `bool`, `type`, `object`, `None`/`NoneType`, `complex`, `range`, `slice`, plus the exception hierarchy. This is the base layer of types you can't avoid touching. Worth knowing that `type` is itself in here and is the metaclass of every class.

**`types`.** The module for "types you have but don't have a literal syntax for." `FunctionType`, `MethodType`, `BuiltinFunctionType`, `LambdaType`, `GeneratorType`, `CoroutineType`, `AsyncGeneratorType`, `ModuleType`, `MappingProxyType` (the read-only dict view you get from `cls.__dict__`), `SimpleNamespace`, `MethodWrapperType`, `WrapperDescriptorType`, `MemberDescriptorType`, `GetSetDescriptorType`. If you've ever wanted to ask "what kind of callable is this?" or introspect a class's machinery, this is where the answers live. Also `types.new_class` and `types.resolve_bases` for runtime class construction.

**`collections.abc`.** The abstract base classes that define Python's structural protocols: `Iterable`, `Iterator`, `Generator`, `Sequence`, `MutableSequence`, `Mapping`, `MutableMapping`, `Set`, `MutableSet`, `Hashable`, `Sized`, `Container`, `Callable`, `Awaitable`, `Coroutine`, `AsyncIterable`, `AsyncIterator`, `Reversible`, `Collection`, `ItemsView`/`KeysView`/`ValuesView`, `MappingView`. These are the "what does it mean to be a sequence / mapping / iterable" definitions. They're also what you use in modern type hints (`Iterable[int]` rather than `List[int]`).

**`collections`.** Concrete specialized containers: `deque`, `Counter`, `OrderedDict`, `defaultdict`, `ChainMap`, `namedtuple`, `UserDict`/`UserList`/`UserString`. Less about "the type system" and more about "the standard concrete types beyond the builtins," but you can't really claim to know Python's data types without these.

**`typing`.** The static type-hint vocabulary: `Any`, `Union`, `Optional`, `Literal`, `Final`, `ClassVar`, `Annotated`, `TypeVar`, `ParamSpec`, `TypeVarTuple`, `Generic`, `Protocol`, `runtime_checkable`, `TypedDict`, `NamedTuple`, `NewType`, `cast`, `overload`, `TYPE_CHECKING`, `Self`, `Never`, `LiteralString`, `Concatenate`, `Unpack`, `Required`/`NotRequired`, `assert_type`, `assert_never`, `reveal_type`, `get_type_hints`, `get_origin`, `get_args`. This is the entire static-typing surface area. A lot of it has migrated to native syntax in modern Python (`list[int]` instead of `List[int]`, `X | Y` instead of `Union[X, Y]`, `type` statement for aliases) but the module is still where the named utilities live.

**`dataclasses`.** `dataclass`, `field`, `fields`, `asdict`, `astuple`, `replace`, `make_dataclass`, `KW_ONLY`, `MISSING`. The decorator-driven way to define record-like classes. Also a great worked example of how Python's runtime introspection enables high-level constructs.

**`enum`.** `Enum`, `IntEnum`, `StrEnum`, `Flag`, `IntFlag`, `auto`, `unique`, `member`, `nonmember`. Enumerated types as proper classes with metaclass machinery. Worth seeing because it's a non-trivial use of metaclasses in the standard library.

**`numbers`.** The numeric tower as ABCs: `Number`, `Complex`, `Real`, `Rational`, `Integral`. Mostly conceptual — most code doesn't `isinstance(x, numbers.Real)` — but it's the official answer to "how does Python organize numeric types."

**`abc`.** `ABC`, `ABCMeta`, `abstractmethod`, `abstractclassmethod`, `abstractstaticmethod`, `abstractproperty`, `update_abstractmethods`. The machinery underlying `collections.abc` and any user-defined abstract base classes. Also where `register()` lives, the mechanism for "virtual subclassing" without inheritance.

**`inspect`.** Not a type module per se, but the introspection toolkit you need to actually _see_ the type system in action: `signature`, `Parameter`, `getmembers`, `isclass`, `isfunction`, `ismethod`, `isgeneratorfunction`, `iscoroutinefunction`, `getmro`, `getsource`, `get_annotations`. If `types` tells you what kinds of things exist, `inspect` lets you ask any given object what kind of thing it is.

**`functools`.** `partial`, `partialmethod`, `singledispatch`, `singledispatchmethod`, `cached_property`, `wraps`, `reduce`, `lru_cache`, `cache`, `total_ordering`. Relevant to types because `singledispatch` is Python's answer to type-based dispatch, and `cached_property` is a worked example of the descriptor protocol you can read.

**`weakref`.** `ref`, `proxy`, `WeakValueDictionary`, `WeakKeyDictionary`, `WeakSet`, `finalize`. Important because it reveals that "holding a reference" is itself part of the object model — and that some types (most builtins like `int`, `str`, `tuple`) can't be weakly referenced at all, which tells you something about how they're implemented.

**`copy`.** `copy`, `deepcopy`, plus the `__copy__`/`__deepcopy__` hooks. Small module, but it's the official answer to "how do values relate to identity" and it interacts with the type system through those dunder hooks.

If you read through `builtins`, `types`, `collections.abc`, `typing`, `dataclasses`, `enum`, `abc`, and `inspect`, you'll have seen essentially every piece of Python's type machinery — both the runtime side (what kinds of objects exist, how classes are built, how protocols work) and the static side (how you describe those types to a checker). The others (`collections`, `numbers`, `functools`, `weakref`, `copy`) round out the picture but aren't strictly about types so much as about types-in-use.

One thing worth flagging: `typing` and the runtime type system are _parallel_ systems, not the same one. `typing.List[int]` and `list[int]` produce objects mainly meant for static analysis tools (mypy, pyright); at runtime they mostly just sit there. The runtime type of `[1, 2, 3]` is `list`, full stop — the `[int]` part is metadata for the type checker, not enforced by the interpreter. Knowing where that line is drawn is itself part of "getting the full picture."

## Type confusion (Iterable)

For several years there were effectively _three_ `Iterable`s coexisting, which was as confusing as it sounds. Here's the timeline:

**Python 2.6 (2008):** `collections.Iterable` is introduced, along with the rest of the container ABCs, sitting directly in `collections`.

**Python 3.3 (2012):** `collections.abc` is created. `collections.abc.Iterable` becomes the canonical location. `collections.Iterable` continues to work as an alias — no warning yet, just a quiet redirect.

**Python 3.5 (2015):** `typing` lands (PEP 484). `typing.Iterable` is introduced as a _generic_ version of the ABC — basically `collections.abc.Iterable` wrapped so you could write `Iterable[int]` for type-checker consumption. At this point you have:

- `collections.Iterable` — old alias, still works
- `collections.abc.Iterable` — canonical runtime ABC, not subscriptable
- `typing.Iterable` — subscriptable for type hints, runtime-equivalent to the ABC

All three referred to the same underlying concept; the difference was ergonomics. If you wanted to do `isinstance(x, Iterable)`, you used `collections.abc.Iterable`. If you wanted to write `def f(xs: Iterable[int])`, you had to use `typing.Iterable`, because the ABC itself didn't support `[int]` until later.

**Python 3.9 (2020):** PEP 585 makes the runtime ABCs themselves subscriptable. Now `collections.abc.Iterable[int]` works directly in type hints, with no `typing` import needed. `typing.Iterable` is officially deprecated (though it still works and probably will for a long time, because removing it would break enormous amounts of code). The two-namespace situation collapses back toward one.

**Python 3.10 (2021):** `collections.Iterable` (the bare alias, no `.abc`) is finally removed. Code that did `from collections import Iterable` breaks.

So the messy middle period — roughly 2015 to 2020 — had genuine triple availability, and a lot of codebases mixed them inconsistently. You'd see one file importing from `collections.abc` for `isinstance` checks, another importing from `typing` for annotations, and sometimes a third still using the deprecated `collections` alias because some old example showed it that way.

The reason `typing.Iterable` had to exist as a separate thing in 3.5 is that `collections.abc.Iterable[int]` would have raised `TypeError` — ABCs weren't subscriptable then. The `typing` module's whole job at first was to provide subscriptable wrappers around existing ABCs and builtins (`typing.List` wrapping `list`, `typing.Dict` wrapping `dict`, etc.) precisely because the underlying types didn't support `[X]`. PEP 585 was the cleanup that retroactively made the wrappers unnecessary, and now we're in a long deprecation tail where `typing.List`, `typing.Dict`, `typing.Iterable`, and friends still work but the recommended forms are `list`, `dict`, and `collections.abc.Iterable`.

The lesson, if there is one, is that Python's type system was bolted onto the runtime gradually, and each addition had to live alongside the previous arrangement until enough time passed to deprecate the old way. The duplication wasn't a design choice — it was the cost of adding a static type layer to a language that didn't have one for 25 years.